# Chapter 3 &mdash; $\Sigma^*$: the Universal Language

**Concept 2 of the Chapter 3 decomposition:** *The Two Roles of Star, and $\Sigma^*$*

Star both expresses "zero or more repetitions" and <b>constructs the universe</b> $\Sigma^*$ &mdash; the missing ingredient for complementation.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter3/Concept-Sigma-Star-Universal-Language/Concept-Sigma-Star-Universal-Language.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
# Run me first.  Works on Colab and on a local Jove checkout.
import os, subprocess, sys

def _git(*a):
    r = subprocess.run(('git',) + a, capture_output=True, text=True)
    return r.stdout.strip() if r.returncode == 0 else ''

REPO = 'https://github.com/ganeshutah/Jove'
try:                       # ---- Colab: clone once, pull thereafter ----
    import google.colab
    was = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD') if os.path.isdir('Jove') else ''
    if os.path.isdir('Jove') and not was:
        print('Jove: WARNING ./Jove exists but is not a git checkout -- left as is')
    elif was:
        _git('-C', 'Jove', 'pull', '-q', '--ff-only')
        now = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD')
        if now and now != was:
            print('Jove: PULLED  %s -> %s' % (was, now))
            print(_git('-C', 'Jove', 'log', '--oneline', was + '..' + now))
        else:
            print('Jove: PULLED  already current at %s' % (now or was))
    else:
        _git('clone', '-q', REPO, 'Jove')
        print('Jove: CLONED  at %s' % (_git('-C', 'Jove', 'rev-parse',
                                             '--short', 'HEAD') or '?'))
    JOVE = 'Jove'
except ImportError:        # ---- local: the checkout above Chapter<N>/ ----
    JOVE = next((p for p in ('../..', '../../..', '..', '.')
                 if os.path.isdir(os.path.join(p, 'jove'))), '../..')
    print('Jove: LOCAL   checkout at %s'
          % (_git('-C', JOVE, 'rev-parse', '--short', 'HEAD') or '?'))
sys.path.insert(0, JOVE)

# A session can already hold an OLDER jove in sys.modules.  The pull above
# updates the files on disk, but `import` would hand back the cached module --
# so a fixed library still behaves like the broken one.  Drop them first.
for _m in [k for k in list(sys.modules) if k == 'jove' or k.startswith('jove.')]:
    del sys.modules[_m]

from jove.LangDef        import *
from jove.Def_md2mc      import *
from jove.DotBashers     import *
from jove.Def_DFA        import *
from jove.AnimateDFA     import *

import jove; print('Jove loaded from', list(jove.__path__)[0])
import jove.AnimateDFA as _a; print('animation toolbar:',
      'ready' if hasattr(_a.AnimateDFA, '_ipython_display_')
      else 'STALE -- restart the runtime, then re-run')

## 1. The idea


Star plays two roles.

**Repetition**: make $i$ selections from $L$ and concatenate them.

**The universal language**: tap a one-key keyboard finitely often and you get
$\{\varepsilon,1,11,\ldots\} = \{1\}^*$. With two keys you get everything:
$\Sigma^* = \{\varepsilon,0,1,00,01,10,11,000,\ldots\}$.

Beware the contrast with $Nplicate = \{a^i : a \in \Sigma, i \ge 0\}$: there you pick
**one** $a$ and repeat *that*. In $\Sigma^*$ you may **change the selection at every step**.

## 2. Definitions

### $\Sigma^*$ versus $Nplicate$

In [ ]:
Sigma = {'0','1'}

def nplicate(Sigma, n):
    """Pick ONE symbol, repeat it i times. NOT the same as Sigma*."""
    return {a*i for a in Sigma for i in range(n+1)}

### A DFA whose language is all of $\Sigma^*$

One state, initial and final, looping on everything.

In [ ]:
sigmastar = md2mc('''DFA
IF : 0 | 1 -> IF
''')
print("states :", sorted(sigmastar["Q"]), " final :", sorted(sigmastar["F"]))

<!-- nav-strip -->

---

&larr;&nbsp;[Ch3&nbsp;1.&nbsp;Star: Three Equivalent Definitions](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter3/Concept-Star-Three-Definitions/Concept-Star-Three-Definitions.ipynb) &nbsp;&middot;&nbsp; [**Chapter 3** index](https://github.com/ganeshutah/Jove/blob/master/Chapter3/README.md) &nbsp;&middot;&nbsp; [Ch3&nbsp;3.&nbsp;Star Bounded at $n$, and `lstar`](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter3/Concept-Lstar-Bounded/Concept-Lstar-Bounded.ipynb)&nbsp;&rarr;

---

## 3. Tests

$Nplicate$ misses $01$ and $10$ &mdash; it never changes its mind.

In [ ]:
N = nplicate(Sigma, 3)
Sstar = lstar(Sigma, 3)
print("Nplicate :", sorted(N, key=len))
print("missing from Nplicate but in Sigma* :", sorted(lminus(Sstar, N), key=len)[:8])
assert '01' in Sstar and '01' not in N
print()
print("In Sigma* the selection is re-made each step; in Nplicate it is fixed.")

$\Sigma^*$ really is everything, up to the bound.

In [ ]:
for n in range(4):
    print("lstar(Sigma,%d) : %d strings, longest %d"
          % (n, len(lstar(Sigma,n)), max(len(s) for s in lstar(Sigma,n))))
assert len(lstar(Sigma,3)) == 15   # 1+2+4+8

The DFA accepts every string we can throw at it.

In [ ]:
for s in ['', '0', '1', '0101', '1110']:
    print("%-6r accepted by the Sigma* DFA? %s" % (s, accepts_dfa(sigmastar, s)))
assert all(accepts_dfa(sigmastar, s) for s in lstar(Sigma,4))
print("\nAll 31 strings of length <= 4 accepted. This machine IS Sigma*.")

## 4. Animation


The universal machine: one state, and every symbol loops back. There is nothing to
reject, which is precisely what "universal" means.

In [ ]:
from jove.AnimateDFA import *
AnimateDFA(sigmastar, FuseEdges=True)

## 5. Exercises


1. Build the DFA for $\{1\}^*$ over $\Sigma=\{0,1\}$. How many states now, and why?
2. Give a string in $\Sigma^*$ that is in no $Nplicate$ set at all.
3. Why does complementation need $\Sigma^*$ rather than just "all the strings we have
   seen"?

In [ ]:
# Your work for the exercises above.

## 6. Where next

In [ ]:
# Previous / next, and a search box for all 245 concepts.
# Type a chapter (Chapter7, ch7) or words from a title (pumping, subset).
#
# Following a link opens a NEW Colab runtime. To pull another concept's
# definitions into THIS session instead:  load_here('Chapter7/Concept-...')
import os, sys
try:                       # usually already done by the Setup cell
    import jove
except ModuleNotFoundError:
    _p = next((p for p in ('Jove', '../..', '../../..', '..', '.')
               if os.path.isdir(os.path.join(p, 'jove'))), None)
    if _p:
        sys.path.insert(0, _p)
try:
    from jove.Nav import nav, load_here
    nav(here='Chapter3/Concept-Sigma-Star-Universal-Language')
except ModuleNotFoundError:
    print('Jove is not on the path yet.')
    print('Run the Setup cell at the top of this notebook, then re-run this one.')